# Medienbeiträge: von den Rohdaten zum Zeitstrahl

Die Funktionen stammen aus `1_filter_data.py` und, soweit das Paket scipy vorhanden ist, aus `2_detect_peaks.py`.

1. Rohdaten ansehen
2. Filtern und nur projektbezogene Beiträge behalten
3. Die gefilterten Daten zeigen
4. Erst danach nach Medium aufteilen
5. Sprache, als Entwurf
6. Städte über eine erweiterbare Namensliste
7. Themen- und Wortcluster, dazu Kuchendiagramme
8. Zeitstrahl mit Peaks, Ablage unter `peaks/peak1`, `peaks/peak2`, … und die häufigsten Begriffe je Peak


In [ ]:
import importlib.util
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

DATEN_NAME = "FSI_Heilbronn_Medienbeiträge (1).json"


def finde_datei(dateiname):
    hier = Path.cwd().resolve()
    for ordner in (hier, hier.parent, hier.parent.parent):
        kandidat = ordner / dateiname
        if kandidat.is_file():
            return kandidat
    raise FileNotFoundError(dateiname)


def modul_laden(dateiname, modulname):
    pfad = finde_datei(dateiname)
    spec = importlib.util.spec_from_file_location(modulname, pfad)
    modul = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modul)
    return modul, pfad.parent


filter_mod, SKRIPT_ORDNER = modul_laden("1_filter_data.py", "filter_data")
try:
    peak_mod, _ = modul_laden("2_detect_peaks.py", "detect_peaks")
except Exception as fehler:
    peak_mod = None
    print("Peak-Modul nicht geladen, lokale Peak-Suche wird genutzt:", type(fehler).__name__)

ROH_PFAD = finde_datei(DATEN_NAME)
PEAK_ORDNER = SKRIPT_ORDNER / "peaks"
print("Rohdaten:", ROH_PFAD)
print("Skripte:", SKRIPT_ORDNER)

SOCIAL_RELEVANT = frozenset({
    "YouTube", "Instagram", "Facebook", "Mastodon", "Flickr", "Reddit", "LinkedIn", "TikTok", "X",
})
PLATTFORM_DOMAINS = (
    ("YouTube", ("youtube.com", "youtu.be")),
    ("Instagram", ("instagram.com",)),
    ("Facebook", ("facebook.com",)),
    ("Reddit", ("reddit.com",)),
    ("LinkedIn", ("linkedin.com",)),
    ("TikTok", ("tiktok.com",)),
    ("Telegram", ("t.me", "telegram.me", "telegram.org")),
    ("Flickr", ("flickr.com",)),
    ("Pinterest", ("pinterest.com",)),
    ("Tumblr", ("tumblr.com",)),
    ("Bluesky", ("bsky.app",)),
    ("Vimeo", ("vimeo.com",)),
    ("X", ("twitter.com", "x.com")),
)
URLAUB_HOST_MARKEN = (
    "erlebnisrundreisen", "holidaycheck", "urlaubspiraten", "hometravel", "phototravel",
    "alpinforum", "skipass", "ski-nordique", "bergrettung", "22places", "tui.com",
)
REGIONAL_HOST_MARKEN = (
    "meinbezirk", "anzeiger", "rundblick", "24hamburg", "bw24", "sauerlandkurier",
    "salzkammergut", "rheingau", "az-online", "krone.at", "kurier.at", "vol.at",
)

# Neue Städte: ein Eintrag mit Suchmustern genügt. Die Muster gelten für Titel und Text.
STAEDTE = {
    "Heilbronn": [r"heilbronn"],
    "Bonn": [r"\bbonn\b"],
    "Paris": [r"\bparis\b", r"cr[ée]teil", r"[îi]le-de-france", r"ile-de-france", r"val-de-marne"],
}

THEMEN = {
    "Kosten": ["kosten", "million", "euro", "budget", "teuer"],
    "Verkehr": ["öpnv", "oepnv", "nahverkehr", "stau", "bus", "pendler", "verkehr"],
    "Trasse": ["trasse", "strecke", "route", "stadtbild", "gondel"],
    "Politik": ["gemeinderat", "beschluss", "planung", "vorplanung", "stadtrat"],
    "Klima": ["klima", "nachhaltig", "umwelt", "lärm", "laerm"],
    "Campus und KI": ["ipai", "campus", "künstliche", "intelligenz"],
}

STOPPWOERTER = {
    "dass", "daß", "eine", "einer", "einem", "einen", "nicht", "auch", "sich", "sind",
    "wird", "wurde", "werden", "haben", "hat", "nach", "noch", "oder", "aber", "über",
    "unter", "zwischen", "sowie", "beim", "schon", "nur", "wie", "aus", "bei", "mit",
    "von", "für", "und", "der", "die", "das", "den", "dem", "des", "ein", "ist", "auf",
    "als", "zum", "zur", "vom", "im", "in", "am", "an", "zu", "es", "sie", "wir", "man",
    "les", "des", "une", "dans", "pour", "par", "sur", "avec", "plus", "que", "qui",
    "est", "sont", "pas", "aux", "cette", "dans", "leur", "leurs", "nous", "vous",
    "the", "and", "for", "with", "this", "that", "from", "https", "http", "www", "weiterlesen", "artikel", "diese", "dieser", "diesem", "durch", "allen",
}


def host_aus_quelle(quelle):
    host = str(quelle or "").strip().lower()
    return host[4:] if host.startswith("www.") else host


def host_ist(host, domain):
    return host == domain or host.endswith("." + domain)


def plattform_roh(quelle, medientyp):
    host = host_aus_quelle(quelle)
    for name, domains in PLATTFORM_DOMAINS:
        if any(host_ist(host, domain) for domain in domains):
            return name
    if medientyp == "Microblog":
        return "Mastodon"
    if medientyp in {"YouTube Video", "YouTube Comments"}:
        return "YouTube"
    return None


def plattform_bestimmen(quelle, medientyp):
    name = plattform_roh(quelle, medientyp)
    return name if name in SOCIAL_RELEVANT else None


def sprache_label(code):
    code = str(code or "").lower().split("-")[0]
    return {"de": "Deutsch", "fr": "Französisch", "en": "Englisch"}.get(code, code or "unbekannt")


def urlaub_bestimmen(host, medientyp, titel, inhalt):
    text = f"{titel}\n{inhalt}".lower()[:2500]
    if any(marke in host for marke in URLAUB_HOST_MARKEN):
        return True
    if medientyp == "Forums":
        return True
    treffer = sum(w in text for w in ("urlaub", "reise", "wanderung", "skigebiet", "ferien", "hotel"))
    return treffer >= 2


def regional_bestimmen(host, medientyp):
    if medientyp == "News":
        return False
    return any(marke in host for marke in REGIONAL_HOST_MARKEN)


def kanal_bestimmen(medientyp, plattform, host, titel, inhalt):
    if isinstance(plattform, str) and plattform:
        return "Social Media"
    if medientyp == "News":
        return "Zeitungen und klassische Medien"
    if urlaub_bestimmen(host, medientyp, titel or "", inhalt or ""):
        return "Urlaub"
    if regional_bestimmen(host, medientyp):
        return "Regionalnachrichten"
    return "Andere Webquellen"


def staedte_finden(text, regeln=None):
    regeln = STAEDTE if regeln is None else regeln
    text = str(text or "").lower()
    return [stadt for stadt, muster in regeln.items() if any(re.search(m, text) for m in muster)]


def thema_finden(text):
    text = str(text or "").lower()
    treffer = {name: sum(wort in text for wort in woerter) for name, woerter in THEMEN.items()}
    beste = max(treffer, key=treffer.get)
    return beste if treffer[beste] else "ohne Thema"


def tokens(text, limit=400):
    wort = re.findall(r"[a-zäöüßàâéèêëîïôùûç]{4,}", str(text or "").lower()[:limit])
    return [w for w in wort if w not in STOPPWOERTER]


def achsen_tausender(wert, _pos):
    return f"{int(wert):,}".replace(",", ".")


def kuchendiagramm(serie, titel):
    serie = serie[serie > 0].sort_values(ascending=False)
    if serie.empty:
        print(titel, ": keine Daten")
        return
    if len(serie) > 8:
        rest = serie.iloc[7:].sum()
        serie = serie.iloc[:7].copy()
        serie.loc["weitere"] = rest
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    ax.pie(serie.values, labels=[str(x) for x in serie.index], autopct="%1.0f%%", startangle=90)
    ax.set_title(titel)
    fig.tight_layout()
    plt.show()


def tagesreihe(rahmen):
    daten = rahmen.dropna(subset=["datum"]).copy()
    taeglich = daten.groupby("datum").size()
    index = pd.date_range(taeglich.index.min(), taeglich.index.max(), freq="D")
    return taeglich.reindex(index, fill_value=0)


def peak_indizes(werte):
    """Tage, die klar über dem üblichen Niveau liegen, mindestens sieben Tage auseinander."""
    werte = np.asarray(werte, dtype=float)
    if len(werte) < 14 or float(np.std(werte)) == 0:
        return np.array([], dtype=int)
    abstand = getattr(peak_mod, "MIN_PEAK_DISTANCE_DAYS", 7) if peak_mod else 7
    mindest = max(10.0, float(np.median(werte) + 2.5 * np.std(werte)))
    kandidaten = [
        i for i in range(1, len(werte) - 1)
        if werte[i] >= werte[i - 1] and werte[i] >= werte[i + 1] and werte[i] >= mindest
    ]
    gewaehlt = []
    for i in sorted(kandidaten, key=lambda j: werte[j], reverse=True):
        if all(abs(i - alt) >= abstand for alt in gewaehlt):
            gewaehlt.append(i)
    return np.array(sorted(gewaehlt), dtype=int)

PROJEKT_REGELN = {
    name: {
        "topic_terms": list(regeln["topic_terms"]),
        "context_terms": list(regeln["context_terms"]),
    }
    for name, regeln in filter_mod.PROJECT_RULES.items()
}
PROJEKT_ORDNER_NAMEN = {
    "Heilbronn_Seilbahn": "Heilbronn",
    "Bonn_Seilbahn": "Bonn",
    "Paris_Cable_C1": "Paris",
    "IPAI_Campus": "IPAI",
}
PROJEKT_ORDNER = SKRIPT_ORDNER / "projekte"


def projekte_aus_text(text, regeln=None):
    """Projekt, wenn Thema und Kontext zusammen vorkommen. regeln kann neue Projekte enthalten."""
    regeln = PROJEKT_REGELN if regeln is None else regeln
    text = str(text or "").lower()
    treffer = []
    for name, rules in regeln.items():
        thema = any(re.search(muster, text) for muster in rules["topic_terms"])
        kontext = any(re.search(muster, text) for muster in rules["context_terms"])
        if thema and kontext:
            treffer.append(name)
    return treffer


def projektordner_name(schluessel):
    return PROJEKT_ORDNER_NAMEN.get(schluessel, schluessel)


def projekte_filtern(rahmen, regeln=None):
    """Beliebige Tabelle mit title und content auf die Projektregeln filtern."""
    rahmen = rahmen.copy()
    inhalt = rahmen["content"] if "content" in rahmen.columns else ""
    text = rahmen["title"].fillna("").astype(str) + "\n" + pd.Series(inhalt, index=rahmen.index).fillna("").astype(str)
    rahmen["projekte"] = text.map(lambda wert: projekte_aus_text(wert, regeln))
    return rahmen[rahmen["projekte"].map(bool)].copy()




## 1. Unstrukturierte Daten ansehen


In [ ]:
with open(ROH_PFAD, encoding="utf-8") as datei:
    roh_liste = filter_mod.extract_records(json.load(datei))

df_roh = pd.DataFrame(roh_liste)
df_roh["datum"] = pd.to_datetime(df_roh["published_at"], utc=True, errors="coerce").dt.tz_localize(None).dt.floor("D")
print(f"Rohbeiträge: {len(df_roh)}")
print("Zeitraum:", df_roh["datum"].min(), "bis", df_roh["datum"].max())
print("\nSprachen:")
print(df_roh["language_code"].fillna("leer").value_counts().head(12).to_string())
print("\nMedientypen:")
print(df_roh["media_type_name"].value_counts().to_string())
df_roh[["published_at", "language_code", "media_type_name", "source_name", "title"]].head(8)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
df_roh["language_code"].fillna("leer").value_counts().head(8).plot(kind="barh", ax=axes[0], color="#3d7ec9")
axes[0].set_title("Rohdaten: Sprache")
df_roh["media_type_name"].value_counts().plot(kind="barh", ax=axes[1], color="#e07a3d")
axes[1].set_title("Rohdaten: Medientyp")
fig.tight_layout()
plt.show()

monat = df_roh.dropna(subset=["datum"]).groupby(df_roh["datum"].dt.to_period("M")).size()
fig, ax = plt.subplots(figsize=(11, 3.6))
monat.plot(ax=ax, color="#555555")
ax.set_title("Rohdaten: Beiträge pro Monat")
ax.set_ylabel("Beiträge")
fig.tight_layout()
plt.show()



## 2. Filtern

`evaluate_post` aus `1_filter_data.py` prüft Datum, Sprache, Dubletten, Projektbezug und sortiert Tourismus- und Einsatzmeldungen aus.


In [ ]:
gesehen_ids = set()
gesehen_hashes = set()
behalten = []
gruende = Counter()
for beitrag in roh_liste:
    gueltig, projekte, grund = filter_mod.evaluate_post(beitrag, gesehen_ids, gesehen_hashes)
    gruende[grund] += 1
    if gueltig:
        zeile = dict(beitrag)
        zeile["matched_projects"] = projekte
        behalten.append(zeile)

df = pd.DataFrame(behalten)
df["datum"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce").dt.tz_localize(None).dt.floor("D")
df["sprache"] = df["language_code"].map(sprache_label)
df["text"] = (df["title"].fillna("") + "\n" + df["content"].fillna("")).str.slice(0, 2500)
print(f"Behalten: {len(df)} von {len(df_roh)} ({len(df) / len(df_roh):.1%})")
print("\nAusschlussgründe:")
for grund, anzahl in gruende.most_common():
    if grund != "kept":
        print(f"  {anzahl:6d}  {grund}")
print("\nProjekte:")
print(df["matched_projects"].explode().value_counts().to_string())



## Projekte: Heilbronn, Bonn, Paris, IPAI

Ein Beitrag gehört zu einem Projekt, wenn ein Themenwort und ein Kontextwort zusammen vorkommen. Die Regeln starten mit denen aus `1_filter_data.py`. Ein neues oder noch unbekanntes Projekt kommt als weiterer Eintrag in `PROJEKT_REGELN` dazu, ohne den Dateinamen zu ändern. `projekte_filtern` lässt sich auf jede Tabelle mit `title` und `content` anwenden.

Die Treffer liegen unter `projekte/Heilbronn`, `projekte/Bonn`, `projekte/Paris` und `projekte/IPAI`.

In [ ]:
# Neues Projekt auf weiteren Daten, zum Beispiel:
# PROJEKT_REGELN["Stuttgart_Seilbahn"] = {
#     "topic_terms": ["seilbahn"],
#     "context_terms": ["stuttgart"],
# }
# projekte_filtern(neue_tabelle)   # Tabelle braucht die Spalten title und content

df["projekte"] = (
    df["title"].fillna("").astype(str) + "\n" + df["content"].fillna("").astype(str)
).map(projekte_aus_text)
print(df["projekte"].explode().value_counts().to_string())

def beitragszeilen(rahmen):
    export = rahmen.copy()
    export["projekte_text"] = export["projekte"].map(lambda werte: ", ".join(werte) if isinstance(werte, list) else "")
    export["snippet"] = export["content"].fillna("").astype(str).str.slice(0, 400)
    spalten = [
        "mention_id", "published_at", "source_name", "title", "url",
        "language_code", "media_type_name", "projekte_text", "snippet",
    ]
    return export[[spalte for spalte in spalten if spalte in export.columns]]

if PROJEKT_ORDNER.exists():
    for alter in list(PROJEKT_ORDNER.iterdir()):
        if alter.is_dir():
            for datei in alter.glob("*"):
                datei.unlink()
            alter.rmdir()
PROJEKT_ORDNER.mkdir(parents=True, exist_ok=True)

uebersicht_projekte = []
for schluessel, regeln in PROJEKT_REGELN.items():
    name = projektordner_name(schluessel)
    teil = df[df["projekte"].map(lambda werte, ziel=schluessel: isinstance(werte, list) and ziel in werte)]
    ordner = PROJEKT_ORDNER / name
    ordner.mkdir(parents=True, exist_ok=True)
    beitragszeilen(teil).to_json(ordner / "beitraege.json", orient="records", force_ascii=False, indent=2)
    meta = {
        "projekt": name,
        "schluessel": schluessel,
        "beitraege": int(len(teil)),
        "topic_terms": regeln["topic_terms"],
        "context_terms": regeln["context_terms"],
    }
    (ordner / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    uebersicht_projekte.append(meta)
    print(f"{name}: {len(teil)} Beiträge -> {ordner.name}/")

(PROJEKT_ORDNER / "uebersicht.json").write_text(
    json.dumps(uebersicht_projekte, ensure_ascii=False, indent=2), encoding="utf-8"
)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
pd.Series({eintrag["projekt"]: eintrag["beitraege"] for eintrag in uebersicht_projekte}).plot(kind="barh", ax=ax, color="#3d7ec9")
ax.set_title("Gefilterte Daten nach Projekt")
ax.set_xlabel("Beiträge")
fig.tight_layout()
plt.show()
print("Abgelegt unter:", PROJEKT_ORDNER)

## 3. Gefilterte Daten erkunden


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
pd.Series({"Rohdaten": len(df_roh), "Gefiltert": len(df)}).plot(kind="bar", ax=axes[0], color=["#bbbbbb", "#e07a3d"])
axes[0].set_title("Vorher / nachher")
axes[0].tick_params(axis="x", rotation=0)
df["matched_projects"].explode().value_counts().plot(kind="barh", ax=axes[1], color="#3d7ec9")
axes[1].set_title("Gefilterte Daten: Projekte")
fig.tight_layout()
plt.show()

taeglich = tagesreihe(df)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(taeglich.index, taeglich.values, color="#333333", linewidth=1.2)
ax.set_title("Gefilterte Daten: Beiträge pro Tag")
ax.set_ylabel("Beiträge")
fig.tight_layout()
plt.show()



## 4. Aufteilung nach Medium

Erst auf den gefilterten Beiträgen. Social Media enthält nur die relevanten Plattformen.


In [ ]:
df["plattform"] = [plattform_bestimmen(q, m) for q, m in zip(df["source_name"], df["media_type_name"])]
df["kanal"] = [
    kanal_bestimmen(m, p, host_aus_quelle(q), t, c)
    for m, p, q, t, c in zip(df["media_type_name"], df["plattform"], df["source_name"], df["title"], df["content"])
]
print("Kanal:")
print(df["kanal"].value_counts().to_string())
print("\nMedientyp:")
print(df["media_type_name"].value_counts().to_string())
print("\nRelevante Social-Media-Quellen:")
print(df.loc[df["kanal"].eq("Social Media"), "plattform"].value_counts().to_string())
kuchendiagramm(df["kanal"].value_counts(), "Gefilterte Daten: Kanal")
kuchendiagramm(df["media_type_name"].value_counts(), "Gefilterte Daten: Medientyp")



## 5. Aufteilung nach Sprache

Entwurf aus `language_code`. Regionale Codes wie `de-DE` werden auf `Deutsch` zusammengezogen.


In [ ]:
print("Entwurf Sprache (aus language_code, auf die Hauptsprache gekürzt):")
print(df["sprache"].value_counts().to_string())
kuchendiagramm(df["sprache"].value_counts(), "Gefilterte Daten: Sprache")
sprachen_zeit = (
    df.dropna(subset=["datum"])
    .groupby([df["datum"].dt.to_period("M"), "sprache"])
    .size()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(11, 3.8))
sprachen_zeit.plot(ax=ax)
ax.set_title("Entwurf: Beiträge pro Monat nach Sprache")
ax.set_ylabel("Beiträge")
fig.tight_layout()
plt.show()



## 6. Aufteilung nach Stadt

Neue Städte kommen als weiterer Eintrag in `STAEDTE` dazu, jeweils mit einem oder mehreren Suchmustern. Die Liste hängt nicht an einem festen Dateinamen.


In [ ]:
df["staedte"] = df["text"].map(staedte_finden)
df["stadt"] = df["staedte"].map(lambda namen: namen[0] if len(namen) == 1 else ("mehrere Städte" if namen else "ohne Stadt"))
print("Eine Zeile kann mehrere Städte enthalten. Für das Diagramm zählt die erste eindeutige Stadt.")
print(df["stadt"].value_counts().to_string())
print("\nMehrfachnennung (eine Zeile zählt in jeder gefundenen Stadt):")
print(df["staedte"].explode().dropna().value_counts().to_string())
kuchendiagramm(df["stadt"].value_counts(), "Gefilterte Daten: Stadt")



## 7. Cluster und Kuchendiagramme

Die Themencluster nutzen feste Begriffslisten. Die Wortcluster sind ein Entwurf: jeder Beitrag wird über seine Wörter einer von fünf Gruppen zugeordnet.


In [ ]:
df["thema"] = df["text"].map(thema_finden)
print("Themencluster über feste Begriffslisten:")
print(df["thema"].value_counts().to_string())
kuchendiagramm(df["thema"].value_counts(), "Themencluster")

wort_zaehler = Counter()
for text in df["text"]:
    wort_zaehler.update(set(tokens(text)))
vokabular = [wort for wort, _ in wort_zaehler.most_common(80)]
index = {wort: i for i, wort in enumerate(vokabular)}
matrix = np.zeros((len(df), len(vokabular)), dtype=float)
for zeile, text in enumerate(df["text"]):
    for wort in set(tokens(text)):
        if wort in index:
            matrix[zeile, index[wort]] = 1.0
norm = np.linalg.norm(matrix, axis=1, keepdims=True)
matrix = matrix / np.clip(norm, 1e-9, None)

k = 5
zufall = np.random.default_rng(7)
zentren = matrix[zufall.choice(len(matrix), size=k, replace=False)]
labels = np.zeros(len(matrix), dtype=int)
for _ in range(12):
    abstand = ((matrix[:, None, :] - zentren[None, :, :]) ** 2).sum(axis=2)
    labels = abstand.argmin(axis=1)
    for gruppe in range(k):
        if np.any(labels == gruppe):
            zentren[gruppe] = matrix[labels == gruppe].mean(axis=0)

df["wortcluster"] = [f"Cluster {n + 1}" for n in labels]
print("\nAutomatische Wortcluster (Entwurf, häufigste Begriffe):")
for gruppe in range(k):
    gewichte = zentren[gruppe]
    begriffe = [vokabular[i] for i in np.argsort(gewichte)[-6:][::-1]]
    print(f"  Cluster {gruppe + 1} ({int((labels == gruppe).sum())} Beiträge): {', '.join(begriffe)}")
kuchendiagramm(df["wortcluster"].value_counts(), "Wortcluster")



## 8. Zeitstrahl mit Peaks

Zuerst der gesamte gefilterte Verlauf. Ein Peak ist ein lokales Maximum, das mindestens zehn Beiträge hat und klar über dem üblichen Niveau liegt (Median plus das Zweieinhalbfache der Streuung). Zwischen zwei Peaks liegen mindestens sieben Tage, derselbe Abstand wie in `2_detect_peaks.py`.

Die Beiträge im Fenster von drei Tagen davor bis zwei Tage danach liegen unter `peaks/peak1`, `peaks/peak2` und so weiter. Darunter stehen die häufigsten Begriffe dieses Fensters.


In [ ]:
reihe = tagesreihe(df)
peaks = peak_indizes(reihe.values)
print(f"Peaks im Gesamtverlauf: {len(peaks)}")

fig, ax = plt.subplots(figsize=(11.5, 4.4))
ax.plot(reihe.index, reihe.values, color="#333333", linewidth=1.3, label="Beiträge pro Tag")
ax.fill_between(reihe.index, reihe.values, color="#333333", alpha=0.08)
for nummer, idx in enumerate(peaks, start=1):
    ax.axvline(reihe.index[idx], color="#e07a3d", alpha=0.85, linewidth=1.2)
    ax.scatter(reihe.index[idx], reihe.iloc[idx], color="#e07a3d", s=36, zorder=3)
    ax.annotate(
        f"P{nummer}",
        (reihe.index[idx], reihe.iloc[idx]),
        textcoords="offset points",
        xytext=(4, 6),
        fontsize=9,
    )
ax.set_title("Gefilterte Daten: Zeitstrahl mit Peaks")
ax.set_ylabel("Beiträge pro Tag")
ax.set_xlabel("Datum")
fig.tight_layout()
plt.show()

if PEAK_ORDNER.exists():
    for alter in PEAK_ORDNER.glob("peak*"):
        if alter.is_dir():
            for datei in alter.glob("*"):
                datei.unlink()
            alter.rmdir()
PEAK_ORDNER.mkdir(parents=True, exist_ok=True)

uebersicht = []
for nummer, idx in enumerate(peaks, start=1):
    tag = reihe.index[idx]
    fenster = df[df["datum"].between(tag - pd.Timedelta(days=3), tag + pd.Timedelta(days=2))].copy()
    ordner = PEAK_ORDNER / f"peak{nummer}"
    ordner.mkdir(parents=True, exist_ok=True)
    begriffe = Counter()
    for text in fenster["text"]:
        begriffe.update(tokens(text, limit=800))
    top = begriffe.most_common(15)
    meta = {
        "peak": nummer,
        "datum": tag.strftime("%Y-%m-%d"),
        "beitraege_am_tag": int(reihe.iloc[idx]),
        "beitraege_im_fenster": int(len(fenster)),
        "fenster": {
            "start": (tag - pd.Timedelta(days=3)).strftime("%Y-%m-%d"),
            "ende": (tag + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
        },
        "haeufigste_begriffe": [{"begriff": wort, "anzahl": anzahl} for wort, anzahl in top],
    }
    export = fenster.assign(
        snippet=fenster["content"].fillna("").str.slice(0, 400),
        projekte=fenster["matched_projects"].map(lambda werte: ", ".join(werte) if isinstance(werte, list) else ""),
    )[
        ["mention_id", "published_at", "source_name", "title", "url", "language_code", "media_type_name", "projekte", "snippet"]
    ]
    export.to_json(ordner / "beitraege.json", orient="records", force_ascii=False, indent=2)
    (ordner / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    uebersicht.append(meta)
    print(f"\nPeak {nummer} — {meta['datum']} — {meta['beitraege_am_tag']} Beiträge am Tag, {meta['beitraege_im_fenster']} im Fenster")
    print("Häufigste Begriffe:", ", ".join(f"{wort} ({anzahl})" for wort, anzahl in top[:10]))

(PEAK_ORDNER / "uebersicht.json").write_text(json.dumps(uebersicht, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nAbgelegt unter: {PEAK_ORDNER}")

